<a href="https://colab.research.google.com/github/VAdarshReddy28/MLA0204-FOML/blob/main/Practical%20Problem%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import math
import os

# Define a class for the decision tree node
class DecisionTreeNode:
    def __init__(self, attribute=None, label=None, branches=None):
        self.attribute = attribute  # the attribute used to split the data
        self.label = label  # the label assigned to this node
        self.branches = branches if branches is not None else {}  # the branches of the decision tree

# Define a function to calculate the entropy of a dataset
def entropy(data):
    target = data['target']
    n = len(target)
    unique, counts = np.unique(target, return_counts=True)
    entropy_val = 0
    for i in range(len(unique)):
        p = counts[i] / n
        entropy_val -= p * math.log2(p)
    return entropy_val

# Define a function to calculate the information gain of an attribute
def information_gain(data, attribute):
    n = len(data)
    values = data[attribute].unique()
    entropy_s = entropy(data)
    entropy_attr = 0
    for value in values:
        subset = data[data[attribute] == value]
        subset_n = len(subset)
        subset_entropy = entropy(subset)
        entropy_attr += subset_n / n * subset_entropy
    return entropy_s - entropy_attr

# Define the ID3 algorithm
def id3(data, attributes):
    target = data['target']
    # If all the examples have the same target value, return a leaf node with that value
    if len(target.unique()) == 1:
        return DecisionTreeNode(label=target.iloc[0])
    # If there are no attributes left to split on, return a leaf node with the most common target value
    if len(attributes) == 0:
        return DecisionTreeNode(label=target.value_counts().idxmax())
    # Otherwise, select the attribute with the highest information gain
    gains = {attr: information_gain(data, attr) for attr in attributes}
    best_attribute = max(gains, key=gains.get)
    # Create a new decision tree node with the selected attribute
    node = DecisionTreeNode(attribute=best_attribute)
    # Split the data based on the selected attribute and recursively build the tree
    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value].drop(best_attribute, axis=1)
        if len(subset) == 0:
            node.branches[value] = DecisionTreeNode(label=target.value_counts().idxmax())
        else:
            new_attributes = attributes.copy()
            if best_attribute in new_attributes:
                new_attributes.remove(best_attribute)
            node.branches[value] = id3(subset, new_attributes)
    return node

# Helper function to print the custom tree structure beautifully
def print_tree(node, depth=0):
    indent = "  " * depth
    if node.label is not None:
        print(f"{indent}Predict: {node.label}")
    else:
        print(f"{indent}[Split on attribute: {node.attribute}]")
        for value, branch in node.branches.items():
            print(f"{indent}  Value: {value} ->")
            print_tree(branch, depth + 2)

# Load the dataset
file_path = '/content/play_tennis (1).csv'
if not os.path.exists(file_path):
    file_path = 'play_tennis.csv'

data = pd.read_csv(file_path)

# Ensure proper column mapping for target variable
if 'Target' in data.columns:
    data = data.rename(columns={'Target': 'target'})
elif 'play' in data.columns:
    data = data.rename(columns={'play': 'target'})

# Exclude index-like columns if present
attributes = [col for col in data.columns if col not in ['target', 'day', 'Day', 'id', 'ID']]

# Print initial info
print(f"Total rows loaded: {len(data)}")
print(f"Initial dataset entropy: {entropy(data):.4f}\n")

# Build the decision tree using ID3 algorithm
root = id3(data, attributes)

print("=== DECISION TREE STRUCTURE FROM YOUR CODE ===")
print_tree(root)

Total rows loaded: 14
Initial dataset entropy: 0.9403

=== DECISION TREE STRUCTURE FROM YOUR CODE ===
[Split on attribute: Outlook]
  Value: sunny ->
    [Split on attribute: Humidity]
      Value: high ->
        Predict: no
      Value: normal ->
        Predict: yes
  Value: overcast ->
    Predict: yes
  Value: rain ->
    [Split on attribute: Wind]
      Value: weak ->
        Predict: yes
      Value: strong ->
        Predict: no
